# 🗺️ Consulta IGAC - Descarga de Predios Catastrales

**Herramienta para consultar la Base Catastral Pública del IGAC 2026**

Transforma coordenadas en cualquier CRS, consulta el FeatureServer de IGAC y descarga predios en formato GeoJSON.

---

**Fuente de Datos:**
- Portal: https://datos-abiertos-igac-igac-oit.hub.arcgis.com/
- FeatureServer: https://services2.arcgis.com/RVvWzU3lgJISqdke/arcgis/rest/services/CATASTRO_PUBLICO_31012026/FeatureServer
- Datos actualizados al: 31 enero 2026
- Licencia: Custom License (consultar metadatos)

## 1️⃣ Instalación de Dependencias

In [ ]:
# Instalar librerías necesarias (ejecutar solo si es la primera vez)
import subprocess
import sys

packages = ['geopandas', 'shapely', 'requests', 'folium', 'pandas']

for package in packages:
    try:
        __import__(package)
        print(f"✓ {package} ya instalado")
    except ImportError:
        print(f"Instalando {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])
        print(f"✓ {package} instalado")

## 2️⃣ Importar Librerías

In [ ]:
import geopandas as gpd
import pandas as pd
import requests
import json
import folium
from shapely.geometry import box, mapping, shape
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')

print("✓ Librerías importadas correctamente")

## 3️⃣ Definir Función Principal de Consulta

In [ ]:
class ConsultaIGAC:
    """
    Clase para consultar la Base Catastral Pública del IGAC
    """
    
    # FeatureServer oficial de IGAC
    BASE_URL = "https://services2.arcgis.com/RVvWzU3lgJISqdke/arcgis/rest/services/CATASTRO_PUBLICO_31012026/FeatureServer"
    
    # Capas disponibles
    LAYERS = {
        7: "U_TERRENO",           # Lotes urbanos (RECOMENDADO)
        14: "R_TERRENO",          # Terrenos rurales
        3: "U_CONSTRUCCION",      # Edificios urbanos
        12: "R_CONSTRUCCION",     # Construcciones rurales
        5: "U_MANZANA",           # Manzanas urbanas
        2: "U_PERIMETRO",         # Perímetro urbano
        8: "U_SECTOR",            # Sectores urbanos
        16: "R_SECTOR",           # Sectores rurales
    }
    
    def __init__(self, timeout=10):
        self.session = requests.Session()
        self.timeout = timeout
        self.resultados = []
        print("✓ ConsultaIGAC inicializada")
    
    def transformar_geometria(self, geometria, crs_origen="EPSG:31818", crs_destino="EPSG:4326"):
        """
        Transforma una geometría de un CRS a otro
        
        Parámetros:
        - geometria: Geometría Shapely o GeoDataFrame
        - crs_origen: CRS original (ej: EPSG:31818 para UTM 18N, EPSG:3857 para Web Mercator)
        - crs_destino: CRS destino (default: EPSG:4326 = WGS84)
        
        Retorna:
        - GeoDataFrame con geometría transformada
        """
        gdf = gpd.GeoDataFrame(geometry=[geometria], crs=crs_origen)
        gdf_transformada = gdf.to_crs(crs_destino)
        return gdf_transformada.geometry[0]
    
    def obtener_bbox(self, geometria):
        """
        Obtiene bounding box de una geometría
        Retorna: (minx, miny, maxx, maxy)
        """
        return geometria.bounds
    
    def consultar_capa(self, layer_id, geometria_wgs84, max_features=2000):
        """
        Consulta una capa específica del FeatureServer
        
        Parámetros:
        - layer_id: ID de la capa (ver LAYERS)
        - geometria_wgs84: Geometría en WGS84 (EPSG:4326)
        - max_features: Número máximo de features a retornar
        
        Retorna:
        - (éxito: bool, data: dict, mensaje: str)
        """
        layer_name = self.LAYERS.get(layer_id, f"Layer_{layer_id}")
        url = f"{self.BASE_URL}/{layer_id}/query"
        
        # Obtener bounding box
        minx, miny, maxx, maxy = self.obtener_bbox(geometria_wgs84)
        
        # Parámetros de consulta
        params = {
            "geometry": json.dumps({
                "xmin": minx, "ymin": miny,
                "xmax": maxx, "ymax": maxy
            }),
            "geometryType": "esriGeometryEnvelope",
            "spatialRel": "esriSpatialRelIntersects",
            "outFields": "*",
            "returnGeometry": "true",
            "f": "geojson",
            "resultRecordCount": str(max_features)
        }
        
        try:
            response = self.session.post(url, data=params, timeout=self.timeout)
            
            if response.status_code == 200:
                data = response.json()
                
                if 'error' in data:
                    error_msg = data['error'].get('message', 'Error desconocido')
                    return (False, None, f"Error servidor: {error_msg}")
                
                features = data.get('features', [])
                if len(features) > 0:
                    return (True, data, f"✓ {len(features)} features encontrados")
                else:
                    return (False, None, "Sin features en zona")
            else:
                return (False, None, f"HTTP {response.status_code}")
        
        except requests.exceptions.Timeout:
            return (False, None, "Timeout (>10s)")
        except Exception as e:
            return (False, None, str(e))
    
    def guardar_geojson(self, data, nombre_archivo=None):
        """
        Guarda datos como GeoJSON
        
        Parámetros:
        - data: GeoJSON dict
        - nombre_archivo: Nombre del archivo (si es None, genera automático)
        
        Retorna:
        - Ruta del archivo guardado
        """
        if nombre_archivo is None:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            feature_count = len(data.get('features', []))
            nombre_archivo = f"predios_{feature_count}features_{timestamp}.geojson"
        
        with open(nombre_archivo, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=2, ensure_ascii=False)
        
        return nombre_archivo
    
    def procesar_zona(self, geometria, crs_origen="EPSG:31818", layers=None, max_features=2000):
        """
        Procesa una zona completa: transforma, consulta y descarga datos
        
        Parámetros:
        - geometria: Geometría en cualquier CRS
        - crs_origen: CRS de la geometría entrada
        - layers: Lista de IDs de capas a consultar (default: U_TERRENO y R_TERRENO)
        - max_features: Máximo de features por capa
        
        Retorna:
        - DataFrame con resumen de resultados
        """
        if layers is None:
            layers = [7, 14]  # U_TERRENO y R_TERRENO por defecto
        
        print(f"\n📍 Procesando zona...")
        print(f"  CRS origen: {crs_origen}")
        
        # Transformar a WGS84
        geom_wgs84 = self.transformar_geometria(geometria, crs_origen, "EPSG:4326")
        bounds = self.obtener_bbox(geom_wgs84)
        print(f"  Zona transformada a WGS84: {bounds}")
        
        # Consultar capas
        self.resultados = []
        
        for layer_id in layers:
            layer_name = self.LAYERS.get(layer_id, f"Layer_{layer_id}")
            print(f"\n  Consultando {layer_name} (Layer {layer_id})...")
            
            success, data, mensaje = self.consultar_capa(layer_id, geom_wgs84, max_features)
            print(f"    {mensaje}")
            
            if success:
                # Guardar GeoJSON
                archivo = self.guardar_geojson(data)
                print(f"    📁 Guardado: {archivo}")
                
                self.resultados.append({
                    'capa': layer_name,
                    'layer_id': layer_id,
                    'features': len(data.get('features', [])),
                    'archivo': archivo,
                    'estado': '✓ Éxito'
                })
        
        return pd.DataFrame(self.resultados)

print("✓ Clase ConsultaIGAC definida")

## 4️⃣ Ejemplo: Consulta para Melgar, Tolima

In [ ]:
# Crear instancia
consulta = ConsultaIGAC()

# Definir grilla en UTM Zone 18N MAGNA-SIRGAS (EPSG:31818)
# Coordenadas: Melgar, Tolima
utm_xmin, utm_ymin = 648000, 954000
utm_xmax, utm_ymax = 653000, 959000

# Crear geometría (bounding box)
grilla_utm = box(utm_xmin, utm_ymin, utm_xmax, utm_ymax)

print(f"\n📦 Grilla definida:")
print(f"  CRS: EPSG:31818 (UTM 18N MAGNA-SIRGAS)")
print(f"  Coordenadas UTM: ({utm_xmin}, {utm_ymin}) a ({utm_xmax}, {utm_ymax})")
print(f"  Área: ~25 km²")

### Procesar la zona

In [ ]:
# Procesar zona (transformar + consultar + descargar)
resultados_df = consulta.procesar_zona(
    geometria=grilla_utm,
    crs_origen="EPSG:31818",
    layers=[7, 14, 3, 12],  # U_TERRENO, R_TERRENO, U_CONSTRUCCION, R_CONSTRUCCION
    max_features=2000
)

# Mostrar resumen
print("\n" + "="*80)
print("RESUMEN DE CONSULTA")
print("="*80)
display(resultados_df)

total_features = resultados_df['features'].sum()
print(f"\n✅ Total de features descargados: {total_features}")

## 5️⃣ Visualizar Datos en Mapa

In [ ]:
# Cargar primer archivo GeoJSON descargado
if len(resultados_df) > 0:
    primer_archivo = resultados_df.iloc[0]['archivo']
    
    # Leer GeoJSON
    gdf = gpd.read_file(primer_archivo)
    
    print(f"\n📊 Primeras filas de datos:")
    print(f"  Archivo: {primer_archivo}")
    print(f"  Features: {len(gdf)}")
    print(f"  Columnas: {list(gdf.columns)[:5]}...")
    print(f"\n  Primeras filas:")
    display(gdf.head())

### Crear mapa interactivo

In [ ]:
if len(resultados_df) > 0 and len(gdf) > 0:
    # Obtener centroide
    centroide = gdf.unary_union.centroid
    
    # Crear mapa
    mapa = folium.Map(
        location=[centroide.y, centroide.x],
        zoom_start=12,
        tiles='OpenStreetMap'
    )
    
    # Agregar geometrías
    for idx, row in gdf.head(100).iterrows():  # Primeras 100 para no saturar
        if row.geometry.geom_type == 'Polygon':
            coords = [[lat, lon] for lon, lat in row.geometry.exterior.coords]
            folium.Polygon(
                coords,
                color='blue',
                weight=1,
                opacity=0.5
            ).add_to(mapa)
    
    # Agregar zona de búsqueda
    grilla_wgs84 = consulta.transformar_geometria(grilla_utm, "EPSG:31818", "EPSG:4326")
    bounds_wgs84 = consulta.obtener_bbox(grilla_wgs84)
    coords_grilla = [
        [bounds_wgs84[1], bounds_wgs84[0]],
        [bounds_wgs84[3], bounds_wgs84[0]],
        [bounds_wgs84[3], bounds_wgs84[2]],
        [bounds_wgs84[1], bounds_wgs84[2]]
    ]
    folium.Polygon(
        coords_grilla,
        color='red',
        weight=3,
        opacity=0.7,
        popup='Zona de búsqueda'
    ).add_to(mapa)
    
    mapa

## 6️⃣ Estadísticas y Análisis

In [ ]:
if len(resultados_df) > 0 and len(gdf) > 0:
    # Calcular áreas
    gdf['area_m2'] = gdf.geometry.area
    gdf['area_ha'] = gdf['area_m2'] / 10000  # Convertir a hectáreas
    
    print(f"\n📈 Estadísticas de áreas:")
    print(f"  Área total: {gdf['area_ha'].sum():.2f} ha")
    print(f"  Área promedio: {gdf['area_ha'].mean():.2f} ha")
    print(f"  Área mínima: {gdf['area_ha'].min():.2f} ha")
    print(f"  Área máxima: {gdf['area_ha'].max():.2f} ha")
    
    # Mostrar distribución de áreas
    print(f"\n  Distribución de áreas:")
    print(gdf['area_ha'].describe())

## 7️⃣ Usar con Otras Coordenadas

### Ejemplo: Usar coordenadas en WGS84

In [ ]:
# Para usar WGS84 directamente
from shapely.geometry import box

# Coordenadas WGS84 (Bogotá)
bogota_wgs84 = box(-74.15, 4.55, -74.05, 4.75)

print("Ejemplo para Bogotá:")
print(f"  Grilla WGS84: {bogota_wgs84.bounds}")

# Procesar
# resultados_bogota = consulta.procesar_zona(
#     geometria=bogota_wgs84,
#     crs_origen="EPSG:4326",  # ← Cambiar a WGS84
#     layers=[7],  # Solo U_TERRENO
#     max_features=1000
# )

### Ejemplo: Usar polígono de archivo

In [ ]:
# Para usar un polígono de un shapefile o GeoJSON
# gdf_zona = gpd.read_file('mi_zona.geojson')
# geometria_zona = gdf_zona.unary_union  # Unir todas las geometrías

# resultados = consulta.procesar_zona(
#     geometria=geometria_zona,
#     crs_origen=gdf_zona.crs,  # Usar CRS del archivo
#     layers=[7, 14],
#     max_features=2000
# )

## 8️⃣ Códigos CRS Útiles

| CRS | EPSG | Descripción |
|-----|------|-------------|
| WGS84 (Global) | 4326 | Coordenadas decimales (lat/lon) |
| Web Mercator | 3857 | Usado por Google Maps, OpenStreetMap |
| **UTM 18N MAGNA-SIRGAS** | **31818** | **Colombia Occidental (recomendado para Melgar)** |
| UTM 18N WGS84 | 32618 | Alternativa a MAGNA-SIRGAS |
| Colombia Bogotá | 3116 | Proyección local Bogotá |
| Colombia Gauss Kruger Zona 3 | 3115 | Gauss Kruger Zona 3 (centro) |
| Colombia Gauss Kruger Zona 4 | 3116 | Gauss Kruger Zona 4 (este) |

## 9️⃣ Funciones Auxiliares

In [ ]:
def listar_capas_disponibles():
    """Lista todas las capas disponibles en IGAC"""
    consulta_temp = ConsultaIGAC()
    print("\n🗺️ Capas disponibles en IGAC:")
    print("-" * 50)
    for layer_id, nombre in sorted(consulta_temp.LAYERS.items()):
        print(f"  {layer_id:2d} - {nombre}")

def convertir_utm_a_wgs84(xmin, ymin, xmax, ymax, zona_utm=18, is_norte=True):
    """Convierte coordenadas UTM a WGS84"""
    epsg_utm = f"326{zona_utm}" if is_norte else f"327{zona_utm}"
    geom = box(xmin, ymin, xmax, ymax)
    consulta_temp = ConsultaIGAC()
    geom_wgs84 = consulta_temp.transformar_geometria(geom, f"EPSG:{epsg_utm}", "EPSG:4326")
    bounds = consulta_temp.obtener_bbox(geom_wgs84)
    return bounds  # (minx, miny, maxx, maxy)

# Usar funciones
listar_capas_disponibles()

print("\n📐 Conversión de ejemplo:")
bounds_ejemplo = convertir_utm_a_wgs84(648000, 954000, 653000, 959000)
print(f"  UTM (648000, 954000) a (653000, 959000)")
print(f"  WGS84: {bounds_ejemplo}")

## 🔟 Exportar a Diferentes Formatos

In [ ]:
if len(resultados_df) > 0 and len(gdf) > 0:
    # Exportar a Shapefile
    # gdf.to_file('predios_melgar.shp')
    # print("✓ Shapefile guardado: predios_melgar.shp")
    
    # Exportar a GeoPackage (más eficiente)
    # gdf.to_file('predios_melgar.gpkg', driver='GPKG')
    # print("✓ GeoPackage guardado: predios_melgar.gpkg")
    
    # Exportar a CSV (solo propiedades, sin geometría)
    # gdf.drop('geometry', axis=1).to_csv('predios_melgar.csv')
    # print("✓ CSV guardado: predios_melgar.csv")
    
    print("✓ Formatos de exportación disponibles:")
    print("  - GeoJSON (ya descargado)")
    print("  - Shapefile (.shp)")
    print("  - GeoPackage (.gpkg) ← Recomendado")
    print("  - CSV (.csv)")
    print("\nDescomenta las líneas anteriores para exportar")

---

## 📚 Recursos y Referencias

**Portal IGAC:**
- https://datos-abiertos-igac-igac-oit.hub.arcgis.com/
- ICDE: https://www.icde.gov.co

**Librerías utilizadas:**
- GeoPandas: https://geopandas.org/ (manipulación de datos geoespaciales)
- Shapely: https://shapely.readthedocs.io/ (geometrías)
- Folium: https://python-visualization.github.io/folium/ (mapas interactivos)

**Información sobre CRS:**
- EPSG: https://epsg.io/
- Proyecciones Colombia: https://www.igac.gov.co/

---

**Creado:** 2026-07-27  
**Datos:** Base Catastral Pública del IGAC 01-2026  
**Última actualización:** 31 enero 2026